# Clase 149 — LoRA / QLoRA: fine-tuning eficiente

Implementamos LoRA desde scratch en numpy (CPU-friendly). La API real con HuggingFace + PEFT requiere GPU y se muestra en markdown.

In [ ]:
import numpy as np
rng = np.random.default_rng(42)

# Toy: matriz base W (frozen) 100x100
d_in, d_out = 100, 100
W = rng.standard_normal((d_in, d_out)) * 0.1
print('W shape:', W.shape, '| params:', W.size)

## 1. LoRA desde scratch: `W' = W + B·A`

- `A`: (r, d_out), init normal
- `B`: (d_in, r), init zeros → al inicio, `B·A = 0`, no perturba a W.

In [ ]:
r = 4   # rank
alpha = 8
scale = alpha / r

A = rng.standard_normal((r, d_out)) * 0.01
B = np.zeros((d_in, r))

trainable = A.size + B.size
full = W.size
print(f'trainable (LoRA): {trainable:,} | full FT: {full:,} | ratio: {trainable/full:.2%}')

## 2. Target task: aproximar W_target = W + ΔW con LoRA

Generamos un "ΔW objetivo" low-rank y entrenamos `B·A` para aprenderlo.

In [ ]:
# Target delta: low-rank ground truth
B_true = rng.standard_normal((d_in, r)) * 0.3
A_true = rng.standard_normal((r, d_out)) * 0.3
delta_target = B_true @ A_true
W_target = W + delta_target

# Datos: y = X @ W_target
n = 256
X = rng.standard_normal((n, d_in))
Y = X @ W_target

In [ ]:
# Entreno SOLO A, B (W frozen). Forward: y_hat = X @ (W + scale*B@A)
lr = 0.01
losses = []
for epoch in range(300):
    delta = scale * (B @ A)
    Y_hat = X @ (W + delta)
    err = Y_hat - Y
    loss = (err**2).mean()
    losses.append(loss)
    # Gradientes (regla de cadena)
    grad_delta = 2 * X.T @ err / n   # (d_in, d_out)
    grad_B = scale * grad_delta @ A.T
    grad_A = scale * B.T @ grad_delta
    B -= lr * grad_B
    A -= lr * grad_A

print(f'loss inicial: {losses[0]:.4f} | final: {losses[-1]:.6f}')
print(f'||B@A - B_true@A_true||: {np.linalg.norm(scale*B@A - delta_target):.4f}')

## 3. Inspección: trainable params vs full fine-tuning

In [ ]:
for r_test in [1, 4, 8, 16, 32, 64]:
    tp = r_test * (d_in + d_out)
    print(f'  r={r_test:3d} | trainable={tp:6,} | ratio={tp/full:.2%}')

## 4. QLoRA conceptual: cuantización NF4

QLoRA congela W en 4-bit (NF4 = NormalFloat) y mantiene LoRA en fp16.

**Ahorro de memoria sobre Llama 7B:**

In [ ]:
n_params = 7e9
fp16_gb = n_params * 2 / 1e9          # 14 GB
nf4_gb  = n_params * 0.5 / 1e9        # 3.5 GB (4 bits = 0.5 bytes)
lora_params = 2 * 16 * 4096 * 32      # r=16 sobre ~32 attn layers de 4096
lora_gb = lora_params * 2 / 1e9

print(f'fp16 full:      {fp16_gb:5.2f} GB')
print(f'NF4 base:       {nf4_gb:5.2f} GB')
print(f'LoRA adapters:  {lora_gb:5.4f} GB')
print(f'QLoRA total:    {nf4_gb + lora_gb:5.2f} GB → cabe en GPU consumer (8-12 GB)')

## 5. API real (HuggingFace + PEFT) — requiere GPU

```python
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                          bnb_4bit_compute_dtype='float16')
model = AutoModelForCausalLM.from_pretrained('mistralai/Mistral-7B-v0.1',
                                              quantization_config=bnb, device_map='auto')
lora = LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj','k_proj','v_proj','o_proj'],
                   lora_dropout=0.05, bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # → ~0.5%
# train con SFTTrainer ...
model = model.merge_and_unload()     # merge para deploy
```

## Ejercicio guiado

1. Variar `r` ∈ {2, 4, 8, 16} y reportar loss final + # params.
2. Probar con W_target NO low-rank (full rank random): ¿cuánto puede aproximar LoRA?
3. Implementar `alpha` scheduling (subir scale gradualmente).
4. Bonus: agregar dropout sobre A·B en forward.

## Conclusiones

- LoRA reduce params trainables ~100-1000x sin pérdida de calidad cuando ΔW es low-rank.
- QLoRA permite fine-tuning de 7B en GPU consumer (8 GB).
- `merge_and_unload()` devuelve modelo normal para deploy sin latencia extra.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README de esta clase. El código que usa librerías pesadas (`transformers` / `torch` / `keras` / `diffusers`) es la **API real** de la industria y se valida por sintaxis (los modelos requieren GPU/descarga). Los **núcleos numéricos** están en numpy puro, son **ejecutables** y se autoverifican con `assert`.

In [ ]:
try:
    import torch  # noqa: F401
    from peft import LoraConfig  # noqa: F401
    _HF = True
except Exception:
    _HF = False
print('peft/torch disponible:', _HF)

### Ejercicio 1 — LoRA básico r=16: trainable params ~0.5% (ejecutable)

In [ ]:
# Nucleo numpy: ratio de parametros entrenables para una capa d x d.
import numpy as np
d = 4096
full = d * d
for r_ in (8, 16, 64):
    trainable = 2 * d * r_          # A (r x d) + B (d x r)
    print(f'r={r_:2d}: LoRA={trainable:,} / full={full:,} = {trainable/full:.3%}')
ratio16 = (2 * d * 16) / full
assert ratio16 < 0.01               # r=16 entrena < 1% de la capa
print('OK: r=16 entrena', f'{ratio16:.3%}', 'de los parametros.')

if _HF:
    from transformers import AutoModelForCausalLM
    from peft import LoraConfig, get_peft_model
    model = AutoModelForCausalLM.from_pretrained('mistralai/Mistral-7B-Instruct-v0.2')
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32,
                target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']))
    model.print_trainable_parameters()

### Ejercicio 2 — QLoRA: base en 4-bit, VRAM ~6 GB

In [ ]:
if _HF:
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    import torch
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(
        'mistralai/Mistral-7B-v0.1', quantization_config=bnb, device_map='auto')
    print('base NF4 cargada (~3.5 GB pesos + activaciones).')
else:
    # ejecutable: memoria fp16 vs NF4 para 7B
    n = 7e9
    print(f'fp16: {n*2/1e9:.1f} GB | NF4: {n*0.5/1e9:.1f} GB (4x menos)')

### Ejercicio 3 — Entrenar con TRL `SFTTrainer`

In [ ]:
if _HF:
    from trl import SFTTrainer, SFTConfig
    trainer = SFTTrainer(model=model, train_dataset=ds,
                         args=SFTConfig(output_dir='out', num_train_epochs=1,
                                        per_device_train_batch_size=4,
                                        learning_rate=2e-4))
    trainer.train()                 # solo se actualizan los adapters LoRA
else:
    print('SFTTrainer(model=peft_model, train_dataset=ds, SFTConfig(lr=2e-4)).train()')

### Ejercicio 4 — Inference con el adapter cargado

In [ ]:
if _HF:
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    base = AutoModelForCausalLM.from_pretrained('mistralai/Mistral-7B-v0.1')
    model = PeftModel.from_pretrained(base, 'out')       # base + adapter
    tok = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-v0.1')
    ids = tok('Instrucción: ...', return_tensors='pt')
    print(tok.decode(model.generate(**ids, max_new_tokens=64)[0]))
else:
    print('PeftModel.from_pretrained(base, adapter_dir) -> generate(...)')

### Ejercicio 5 — Merge del adapter para deploy

In [ ]:
if _HF:
    merged = model.merge_and_unload()       # funde B*A dentro de W
    merged.save_pretrained('mistral-merged') # modelo normal, sin latencia extra
    print('merge_and_unload -> pesos fusionados, deploy como modelo estandar.')
else:
    print("model.merge_and_unload(): W' = W + (alpha/r)*B@A, luego save_pretrained.")